# Presidential speeches -> immigration keyword CSV

Makes `speech_keywords.csv` with one row per speech: president, year, title, speech type, and a count for each keyword.

Run this from the repo root. Each file in `speeches/` is one speech stored as JSON, which Python reads as a dictionary with keys like `president`, `date`, `title` and `transcript`.

In [40]:
import glob
import json
import re

import pandas as pd

## Settings

Change these two to change the CSV. The data has no speech-type field, so the type is guessed from the title: the first phrase below that appears in the title wins.

In [41]:
keywords = ["immigration", 'immigrant','migrant','migration','refugees', 'refugee', 'aliens', 'alien', 'undocumented']

speech_types = {
    "inaugural": "Inaugural Address",
    "fireside chat": "Fireside Chat",
    "state of the union": "State of the Union",
    "annual message": "State of the Union",
    "annual address": "State of the Union",
    "veto": "Veto Message",
    "proclamation": "Proclamation",
    "debate": "Debate",
    "press conference": "Press Conference",
    "news conference": "Press Conference",
    "briefing": "Press Conference",
    "farewell": "Farewell Address",
    "national convention": "Convention Speech",
    "nomination": "Convention Speech",
    "message": "Message",
    "remarks": "Remarks / Statement",
    "statement": "Remarks / Statement",
    "address": "Address / Speech",
    "speech": "Address / Speech",
}

## Go through every speech

In [42]:
rows = []

for filename in sorted(glob.glob("speeches/*.json")):
    with open(filename, encoding="utf-8") as f:
        speech = json.load(f)

    # speech type: the first phrase that appears in the title
    title = speech["title"].lower()
    speech_type = "Other"
    for phrase, label in speech_types.items():
        if phrase in title:
            speech_type = label
            break

    # split the transcript into lowercase words
    text = speech["transcript"].lower()
    text = re.sub(r"<[^>]+>", " ", text)   # remove HTML tags such as <br />
    words = re.findall(r"[a-z]+", text)

    row = {
        "president": speech["president"],
        "year": int(speech["date"][:4]),
        "title": speech["title"],
        "speech_type": speech_type,
    }
    for keyword in keywords:
        row[keyword] = words.count(keyword)

    rows.append(row)

## Save

In [43]:
df = pd.DataFrame(rows)
df.to_csv("speech_keywords.csv", index=False)
df.head()

,president,year,title,speech_type,immigration,immigrant,migrant,migration,refugees,refugee,aliens,alien,undocumented
0,Abraham Lincoln,1852,"July 6, 1852: Eulogy on Henry Clay",Other,0,0,0,0,0,0,0,0,0
1,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,0,0,0,2,0,0,0,0,0
2,Abraham Lincoln,1858,"June 16, 1858: ""A House Divided"" Speech",Address / Speech,0,0,0,0,0,0,0,0,0
3,Abraham Lincoln,1860,"February 27, 1860: Cooper Union Address",Address / Speech,0,0,0,0,0,0,0,0,0
4,Abraham Lincoln,1861,"February 11, 1861: Farewell Address",Farewell Address,0,0,0,0,0,0,0,0,0


In [44]:
df["speech_type"].value_counts()

speech_type
State of the Union     210
Address / Speech       204
Message                158
Remarks / Statement    139
Other                   87
Inaugural Address       60
Veto Message            44
Press Conference        39
Proclamation            38
Fireside Chat           30
Convention Speech       24
Farewell Address        13
Debate                  13
Name: count, dtype: int64

## Optional: a more sophisticated version -- Adding terms

The above cell is the simplest approach. But we can make it more sophisticated.
The cell below is the same loop again, with two additions marked `# NEW`:

- `word_count`, so a count can be turned into a rate per 1,000 words
- word families: `econom*` counts every word that starts with `econom` (*economy*, *economic*, *economics*...)

It saves a second file, `speech_keywords_extended.csv`. Feel free to compare them.

In [45]:
import html                                # NEW

families = ["migra", "immigra"]    # NEW

rows = []

for filename in sorted(glob.glob("speeches/*.json")):
    with open(filename, encoding="utf-8") as f:
        speech = json.load(f)

    # speech type: the first phrase that appears in the title
    title = speech["title"].lower()
    speech_type = "Other"
    for phrase, label in speech_types.items():
        if phrase in title:
            speech_type = label
            break

    # split the transcript into lowercase words
    text = speech["transcript"].lower()
    text = re.sub(r"<[^>]+>", " ", text)   # remove HTML tags such as <br />
    text = html.unescape(text)             # NEW: turn codes like &nbsp; into real characters
    words = re.findall(r"[a-z]+", text)

    row = {
        "president": speech["president"],
        "year": int(speech["date"][:4]),
        "title": speech["title"],
        "speech_type": speech_type,
        "word_count": len(words),          # NEW
    }
    for keyword in keywords:
        row[keyword] = words.count(keyword)
    for family in families:                # NEW
        row[family + "*"] = len([word for word in words if word.startswith(family)])

    rows.append(row)

df_immigration = pd.DataFrame(rows)
df_immigration.to_csv("immigration_keywords_speech.csv", index=False)
df_immigration.head()

,president,year,title,speech_type,word_count,immigration,immigrant,migrant,migration,refugees,refugee,aliens,alien,undocumented,migra*,immigra*
0,Abraham Lincoln,1852,"July 6, 1852: Eulogy on Henry Clay",Other,10814,0,0,0,0,0,0,0,0,0,0,0
1,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,33240,0,0,0,2,0,0,0,0,0,2,0
2,Abraham Lincoln,1858,"June 16, 1858: ""A House Divided"" Speech",Address / Speech,3224,0,0,0,0,0,0,0,0,0,0,0
3,Abraham Lincoln,1860,"February 27, 1860: Cooper Union Address",Address / Speech,13719,0,0,0,0,0,0,0,0,0,0,0
4,Abraham Lincoln,1861,"February 11, 1861: Farewell Address",Farewell Address,152,0,0,0,0,0,0,0,0,0,0,0


### Why add a word count?

Some decades have far more text than others, so raw counts mostly show where the most words are. Dividing by the number of words makes decades comparable. Compare the `war` column with the rate next to it.

In [46]:
by_decade = df_immigration.groupby(df_immigration["year"] // 10 * 10)[["word_count", "immigrant"]].sum()
by_decade["immigrant_per_1000_words"] = by_decade["immigrant"] / by_decade["word_count"] * 1000
by_decade.sort_values("immigrant", ascending=False).head()

,word_count,immigrant,immigrant_per_1000_words
year,,,
2010,262657,18,0.068530
1890,206236,10,0.048488
2000,159787,7,0.043808
1980,269750,7,0.025950
1990,214760,4,0.018625


### Why add word families?

Here's the exact word next to its family, totalled over all speeches. You can see why this matters.

In [47]:
df_immigration[[ "refugees", "aliens", "immigrant", "immigration", "immigra*"]].sum()

refugees        82
aliens         125
immigrant       59
immigration    360
immigra*       617
dtype: int64

## Keep in mind

- Debates and press conferences also contain the words of moderators, opponents and reporters.
- A word family can catch words you did not mean, so check what it matches before trusting the column.

# Trying to look at all speeches with immigra*, migra*, refugee, and alien

In [48]:
matching_speeches = df_immigration[
    (df_immigration["migra*"] > 0) | (df_immigration["immigra*"] > 0) | (df_immigration["refugee"] > 0) | (df_immigration["refugees"] > 0) | (df_immigration["alien"] > 0) | (df_immigration["aliens"] > 0) | (df_immigration["undocumented"] > 0)
]

matching_speeches

,president,year,title,speech_type,word_count,immigration,immigrant,migrant,migration,refugees,refugee,aliens,alien,undocumented,migra*,immigra*
1,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,33240,0,0,0,2,0,0,0,0,0,2,0
5,Abraham Lincoln,1861,"March 4, 1861: First Inaugural Address",Inaugural Address,3632,0,0,0,0,0,0,2,0,0,0,0
7,Abraham Lincoln,1861,"December 3, 1861: First Annual Message",State of the Union,6956,1,0,0,0,0,0,0,0,0,0,1
8,Abraham Lincoln,1862,"December 1, 1862: Second Annual Message",State of the Union,8294,0,0,0,1,0,0,2,0,0,2,0
10,Abraham Lincoln,1863,"August 26, 1863: Public Letter to James Conkling",Other,1696,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,Woodrow Wilson,1916,"September 2, 1916: Speech of Acceptance",Address / Speech,5533,0,0,0,0,0,0,0,2,0,0,0
1039,Woodrow Wilson,1916,"September 3, 1916: Speech Accepting the Democr...",Convention Speech,5533,0,0,0,0,0,0,0,2,0,0,0
1047,Woodrow Wilson,1917,"April 6, 1917: Proclamation 1364",Proclamation,1533,0,0,0,0,0,0,1,19,0,0,0
1050,Woodrow Wilson,1917,"December 4, 1917: Fifth Annual Message",State of the Union,3935,0,0,0,0,0,0,0,5,0,0,0


In [49]:
# Create a total count column for sorting
df_immigration["total_immigration_mentions"] = df_immigration["migra*"] + df_immigration["immigra*"] + df_immigration["refugee"] + df_immigration["alien"] + df_immigration["undocumented"]

# Filter, select columns, and sort by most mentions
immigration_summary = (
    df_immigration[df_immigration["total_immigration_mentions"] > 0]
    [["president", "year", "title", "migra*", "immigra*", "refugee", "alien", "total_immigration_mentions"]]
    .sort_values(by="total_immigration_mentions", ascending=False)
)

immigration_summary

,president,year,title,migra*,immigra*,refugee,alien,total_immigration_mentions
375,George W. Bush,2004,"January 7, 2004: Temporary Worker Program Prop...",0,30,0,0,37
911,Theodore Roosevelt,1905,"December 5, 1905: Fifth Annual Message",1,32,0,3,36
462,Grover Cleveland,1897,"March 2, 1897: Veto Message Regarding Immigrat...",0,28,0,6,34
105,Barack Obama,2013,"January 29, 2013: Remarks on Immigration Reform",0,26,0,0,30
115,Barack Obama,2014,"November 20, 2014: Address to the Nation on Im...",0,23,0,0,30
...,...,...,...,...,...,...,...,...
548,James K. Polk,1848,"August 14, 1848: Message Regarding Slavery in ...",0,0,0,1,1
523,James Buchanan,1860,"December 3, 1860: Fourth Annual Message",0,0,0,1,1
519,James Buchanan,1859,"December 19, 1859: Third Annual Message",1,0,0,0,1
516,James Buchanan,1858,"December 6, 1858: Second Annual Message",0,1,0,0,1


Creating CSV?

In [50]:
# Create the new DataFrame containing only the matching rows
df_filtered_speeches = df_immigration[
    (df_immigration["migra*"] > 0)
    | (df_immigration["immigra*"] > 0)
    | (df_immigration["refugee"] > 0)
    | (df_immigration["refugees"] > 0)
    | (df_immigration["alien"] > 0)
    | (df_immigration["aliens"] > 0)
    | (df_immigration["undocumented"] > 0)
].copy()

# Reset index so the new DataFrame has clean 0, 1, 2... numbering
df_filtered_speeches.reset_index(drop=True, inplace=True)
# create a new csv
df_filtered_speeches.to_csv("filtered_immigration_speeches.csv", index=False)



print(df_filtered_speeches.shape)
# Output will show: (220, number_of_columns)

(255, 17)
